# 00 - Esegui B1, B2, C1 e A1 in fila

Kaggle dà a ogni notebook la sua cartella `/kaggle/working`: se i passi girano come notebook
separati, A1 non vede gli output degli altri. Questo notebook li esegue uno dopo l'altro nella
stessa sessione, così ognuno legge gli output del precedente.

Tutti i passi usano la stessa simulazione (`pipeline/dfl_sim.py`): le partite DFL viste come le
darebbe il tracking fornito dalla gara, con una sola palla che manca anche a gioco vivo. La catena
gira una volta per ogni livello in `SIMS` (facile, medio, difficile): una scelta si tiene solo se
regge su tutti e tre. Vedi `ARCHITETTURA.md`.

| Passo | Modulo | Produce | Usato da |
|---|---|---|---|
| 1. B1 palla morta | `pipeline/deadball.py` | `<match>_deadball.npz` (`p_dead`) + modello | B2, A1 |
| 2. B2 tocco + controllo | `pipeline/touch_control.py --dead auto` | `<match>_rule.npz` (`labels`, `p_home`) + `rule_params.json` | A1 |
| 3. C1 rete sui giocatori (GPU) | `pipeline/player_net.py` | `<match>_net.npz` (`p_home`) + modello | A1 |
| 4. A1 decoder | `pipeline/decoder.py` | `<match>_decoded.npz` + `decoder_params.json` + tabella finale | – |

Output di ogni livello in `dfl_outputs/<livello>`, modelli e parametri in `models/<livello>`.
Tempo indicativo per livello con le griglie complete: 30-45 minuti per B1, B2 e A1; con C1 su GPU T4
aggiungere 45-75 minuti. Con `QUICK = True` molto meno.
Tutti i dataset di input devono essere **privati**. Solo partite DFL: nessun dato di BAR-ATH.

## 0. Codice della repo
Se il codice è caricato come dataset Kaggle (qualunque nome), questa cella non fa nulla.
Altrimenti clona il branch con il token GitHub salvato nei Kaggle Secrets come `GITHUB_TOKEN`
(Add-ons → Secrets) e richiede **Internet attivo**.

In [ ]:
import os, glob, subprocess
BRANCH = "feature/notebooks-dfl"
DEST = "/kaggle/working/Hackaton_Barcellona"
code_ds = glob.glob("/kaggle/input/**/pipeline/decoder.py", recursive=True)
if code_ds:
    print("codice trovato come dataset:", os.path.dirname(os.path.dirname(code_ds[0])), "- clone non necessario")
elif os.path.isdir(DEST):
    print("clone gia presente")
elif os.path.isdir("/kaggle/input"):
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://{}@github.com/samugarofalo/Hackaton_Barcellona.git".format(tok)
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, DEST], capture_output=True, text=True)
    print("clone ok" if r.returncode == 0 else r.stderr.replace(tok, "***"))
else:
    print("non su Kaggle: clone non necessario")


## 1. Setup
Stessi input dei singoli notebook: codice della repo, partite DFL convertite (`DFL-MAT-*.npz`). Vedi `notebooks/README.md`.

In [ ]:
import os, sys, subprocess, time, glob
_found = sorted(glob.glob("/kaggle/input/**/pipeline/decoder.py", recursive=True))  # codice caricato come dataset, ovunque sia montato
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] + [os.path.dirname(os.path.dirname(f)) for f in _found] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]
print("repo:", REPO)

QUICK = False   # True = griglie ridotte, per una prova veloce
SIMS = ("easy", "medium", "hard")   # livelli della simulazione (pipeline/dfl_sim.py)
RUN_C1 = True   # rete sui giocatori: serve Accelerator = GPU T4 (su CPU e molto lenta)

# Partite DFL: se ci sono solo gli XML originali, convertili una volta in /kaggle/working
if not os.environ.get("DFL_CONVERTED") and not glob.glob("/kaggle/input/**/DFL-MAT-*.npz", recursive=True):
    xml = sorted(glob.glob("/kaggle/input/**/DFL_04_03_positions_raw_observed_*.xml", recursive=True))
    if xml:
        os.environ["DFL_SRC"] = os.path.dirname(xml[0])
        os.environ["DFL_CONVERTED"] = "/kaggle/working/dfl_converted"
        print("converto", len(xml), "partite da", os.environ["DFL_SRC"], "(qualche minuto)...", flush=True)
        r = subprocess.run([sys.executable, os.path.join(REPO, "pipeline", "convert_dfl.py")], cwd=REPO, env=os.environ.copy(), capture_output=True, text=True)
        print(r.stdout[-3000:], r.stderr[-3000:])

import dfl_paths
print("partite DFL:", len(dfl_paths.match_paths()), "in", dfl_paths.converted_dir())
# i sottoprocessi devono scrivere e leggere nelle stesse cartelle, una per livello
os.environ["DFL_CONVERTED"] = dfl_paths.converted_dir()
OUT_BASE, MODELS_BASE = dfl_paths.out_dir(), dfl_paths.models_dir()
print("output in:", OUT_BASE, "| modelli in:", MODELS_BASE)

In [ ]:
def run(script, *args):
    cmd = [sys.executable, os.path.join(REPO, "pipeline", script), *args]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.run(cmd, cwd=REPO, env=os.environ.copy(), capture_output=True, text=True)
    print(p.stdout[-20000:])
    if p.returncode != 0:
        print(p.stderr[-5000:])
        raise RuntimeError("{} fallito (codice {})".format(script, p.returncode))
    print("--- {} finito in {:.1f} min".format(script, (time.time() - t) / 60))
    print()

def set_level(sim):
    os.environ["DFL_SIM"] = sim
    os.environ["DFL_OUT"] = os.path.join(OUT_BASE, sim)
    for k in ("MTAH_MODELS", "DEADBALL_MODELS", "PLAYER_NET_MODELS"):
        os.environ[k] = os.path.join(MODELS_BASE, sim)

## 2. B1, B2, C1 e A1, per ogni livello
- **B1** addestra LightGBM lasciando fuori una partita alla volta e scrive `p_dead`.
- **B2** (`--dead auto`) usa la palla morta di B1 appena scritta.
- **C1** stima chi ha la palla dai giocatori, anche quando la palla manca. Serve Accelerator = GPU T4;
  con `RUN_C1 = False` viene saltato e A1 lavora senza la rete.
- **A1** legge da solo gli output degli altri. Nella riga `sources` deve comparire `p_dead: B1 file`,
  `p_home B2: yes` e, se C1 ha girato, `p_home C1 net: yes`. Il peso della rete è scelto insieme agli
  altri parametri: se viene scelto 0, la rete non aiuta.

In [ ]:
import importlib, decoder
results = {}
for sim in SIMS:
    print("=" * 30, "livello", sim, "=" * 30, flush=True)
    set_level(sim)
    run("deadball.py", *(["--quick"] if QUICK else []))
    run("touch_control.py", "--dead", "auto", *(["--quick"] if QUICK else []))
    if RUN_C1:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        print("device:", dev, "(attiva Accelerator = GPU T4 se dice cpu)")
        run("player_net.py", "--device", dev, *(["--quick"] if QUICK else []))
    importlib.reload(decoder)
    results[sim] = decoder.evaluate_dfl()

## 3. Cosa riportare al team
Solo numeri aggregati su DFL:

- tabella finale di A1 per livello (spellF1 medio, min, max per metodo);
- riga `sources` (quali output sono stati usati) e parametri scelti;
- dai passi B1 e B2: spellF1 della palla morta e di tocco + controllo.

Differenze sotto circa 0,04 di spellF1 sono rumore. Un pezzo entra nella catena finale solo se
migliora in tutti i livelli.

In [ ]:
import pandas as pd
for sim, res in results.items():
    print(sim, "| sources:", res["sources"])
    print(sim, "| parametri scelti nei fold:", sorted(set(res["params"].values())))
pd.concat({sim: res["summary"]["spell_f1"] for sim, res in results.items()}, axis=1)